# 02. Data Cleaning & Cross-Split Leakage

Two goals: remove junk from the training data, and check whether the published
train/dev/test splits are actually independent.

In [1]:
import sys
sys.path.append("../src")
import pandas as pd
from data import load_split, clean_split, remove_leakage, build_processed
from config import FIG_DIR

# raw counts
for s in ["train", "dev", "test"]:
    print(f"raw {s}: {len(load_split(s)):,}")

raw train: 345,846
raw dev: 43,233
raw test: 43,233


## Cleaning

Manual inspection of extreme-scoring jokes surfaced three data-quality issues
that automated pipelines would miss:

1. **Exact duplicates.**
2. **Ultra-short / title-only entries.**
3. **NaN / empty rows.**

`clean_split()` handles all three. The threshold of 5 words was chosen because
a higher cutoff would delete legitimate one-line puns.

In [2]:
# build the cleaned splits and show the result
splits = build_processed(min_words=5)

for name, df in splits.items():
    print(f"cleaned {name}: {len(df):,} rows")
print("\nSample cleaned rows:")
splits["train"].sample(5, random_state=0)

[saved] train: 339,499 rows -> D:\Workbook\Humor Intelligence\data\processed\train.csv
[saved] dev: 41,941 rows -> D:\Workbook\Humor Intelligence\data\processed\dev.csv
[saved] test: 41,957 rows -> D:\Workbook\Humor Intelligence\data\processed\test.csv
cleaned train: 339,499 rows
cleaned dev: 41,941 rows
cleaned test: 41,957 rows

Sample cleaned rows:


,score,joke
300698,0,"Once in an African Village Once, in an African..."
279605,0,What do you get when you cross... ....a Mexica...
73653,2,My friends and I wanted to get Indian food las...
75004,2,Girlfriends are like subway seats... You don't...
17718,0,I've started a business selling boats from my ...


In [3]:
# class balance after cleaning (should barely shift vs raw)
def to_bucket(s):
    if s == 0: return "not funny (0)"
    elif s <= 2: return "mild (1-2)"
    elif s <= 5: return "funny (3-5)"
    else: return "very funny (6+)"

order = ["not funny (0)", "mild (1-2)", "funny (3-5)", "very funny (6+)"]
b = splits["train"]["score"].map(to_bucket).value_counts().reindex(order)
print((b / b.sum()).round(3))

score
not funny (0)      0.344
mild (1-2)         0.424
funny (3-5)        0.190
very funny (6+)    0.042
Name: count, dtype: float64


## Cross-split leakage

A less obvious problem: are the dev/test splits actually independent of
train? Reddit reposts can land in multiple splits, meaning the model would be
evaluated on jokes it has already seen during training. That inflates scores
and invalidates the evaluation.

`remove_leakage()` does an exact-match check of every dev/test joke against
the cleaned training set.

In [4]:
# quantify the leakage BEFORE removal
raw_train = clean_split(load_split("train"), min_words=5)
raw_dev   = clean_split(load_split("dev"),   min_words=5)
raw_test  = clean_split(load_split("test"),  min_words=5)

train_jokes = set(raw_train["joke"])
dev_leaked  = raw_dev[raw_dev["joke"].isin(train_jokes)]
test_leaked = raw_test[raw_test["joke"].isin(train_jokes)]

print(f"dev  leakage: {len(dev_leaked):,} / {len(raw_dev):,}  ({len(dev_leaked)/len(raw_dev)*100:.1f}%)")
print(f"test leakage: {len(test_leaked):,} / {len(raw_test):,}  ({len(test_leaked)/len(raw_test)*100:.1f}%)")
print(f"\nThese rows are exact copies of training jokes.")
print(f"Evaluating on them would partially test memorization, not generalization.")

dev  leakage: 1,044 / 42,985  (2.4%)
test leakage: 1,061 / 43,018  (2.5%)

These rows are exact copies of training jokes.
Evaluating on them would partially test memorization, not generalization.


In [5]:
# what do the leaked jokes look like?
print("Sample leaked test jokes (also in train):\n")
for _, row in test_leaked.head(5).iterrows():
    print(f"  [score={row['score']}] {row['joke'][:120]}...")
    print()

Sample leaked test jokes (also in train):

  [score=2] What do you call a guitar player without a girlfriend? Homeless...

  [score=0] When I die, I hope I go peacefully in my sleep like my grandmother. Not screaming like the... ...43 toddlers she murdere...

  [score=2] What's the similarity between a mobile phone and a clitoris? Both turn on with the touch of a finger and every cunt's go...

  [score=0] How can you tell your acne is really starting to get out of hand? The blind start reading your face....

  [score=2] It’s hard to explain puns to kleptomaniacs They always take things literally....



The leaked rows are ordinary jokes that happen to appear in both train and
test because they were reposted on Reddit. The prior rJokes paper (Weller &
Seppi, 2020) did not remove this overlap. After removal, the cleaned splits
used throughout this project are:

| Split | Raw | After cleaning | After leakage removal |
|---|---|---|---|
| train | 345,846 | 339,499 | 339,499 (reference set) |
| dev | 43,233 | 42,985 | 41,941 |
| test | 43,233 | 43,018 | 41,957 |

The impact of this leakage on reported metrics is quantified in
`05_evaluation.ipynb`, where every model is evaluated on both the clean and
leaked test sets.